In [14]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt
import seaborn as sns

### 1) Open Visum .ver

In [15]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"

In [16]:
#Red base GDL (con links agregados por Johan y TALA)
red_base = os.path.join(folder, "Red Base GDL","RedBase Conectores y Atts", "RedBase 150826 - gpkg3.ver")

import win32com.client
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = win32com.client.constants

### 2) Open shp/gpkg Visum Links with attributes

In [17]:
# Read links from the final corrected shapefile
links_red_final = gpd.read_file(os.path.join(folder, 'red_shapefiles', '11_red_final_filtrada', 'last', 'filtered_final_network_3.gpkg'))
#links_homologados = gpd.read_file(os.path.join(folder, "VisumLinks With TransCAD Atts", "Links atts homologados", "RED_VIAL_PRINCIPAL_CON_CAPACIDADES_LARGA_TODOS_NODOS_CON_CARGA_v5.shp"))
print(f"Links en en el shp/gpkg: {len(links_red_final):,}")

Links en en el shp/gpkg: 605,326


In [18]:
links_red_final.columns

Index(['tsysset', 'highway', 'ageb_df_idx', 'ageb_idx', 'ageb_code',
       'from_node', 'to_node', 'u', 'v', 'key', 'length', 'capacity',
       'num_lanes', 'avg_vel', 'limit_vel', 'pop_dens', 'job_dens', 'dist_gdl',
       'boundary', 'cluster', 'subcluster', 'topo_type', 'topo_filt',
       'main_road', 'connector_end_link', 'network_backbone', 'final_filter',
       'geometry'],
      dtype='object')

In [19]:
#Probablemente son mas que los de Visum porque en Visum se borraron los links del macro que no servian 
# y yo solo le pase a Sebastian los los links nuevos de TALA y los trazos Johan
"""links_red_final["ageb_idx"] = (
    links_red_final["ageb_idx"]
    .fillna(-1)
    .astype(int)
)"""

# Read links from Visum
links_from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "from_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "to_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")]
})
print(f"Links Visum Final: {len(links_from_visum):,}")

# Hacer merge on from node to node with Visum links
cols_from_shp = ["from_node", "to_node", "ageb_df_idx", "ageb_idx", "ageb_code", "topo_filt", "main_road", "connector_end_link", "network_backbone", "final_filter"]
#cols_from_shp = ["FROMNODENO", "TONODENO", "CAP_BPR", "CARRILES_F", "LIM_VEL_FI", "RA_JERARQ", "GRUPO_CAP", "F_CAP_BPR", "RA_CARGA", "NOMBRE_FIN"]
links_from_visum = links_from_visum.merge(
    #links_red_final[["from_node", "to_node", "ageb_idx", "capacity", "num_lanes", "avg_vel", "limit_vel", "pop_dens", "job_dens", "dist_gdl", "boundary", "cluster", "subcluster", "topo_type", "topo_filt", "final_filter"]],
    links_red_final[cols_from_shp],
    on=["from_node", "to_node"],
    how="left"
)

Links Visum Final: 605,326


In [21]:
links_from_visum

,No,from_node,to_node,ageb_df_idx,ageb_idx,ageb_code,topo_filt,main_road,connector_end_link,network_backbone,final_filter
0,1.0,1.0,103492.0,1436.0,1041,1410100011041,1,7.0,1,0,1
1,1.0,103492.0,1.0,1436.0,1041,1410100011041,1,NaN,1,0,1
2,2.0,1.0,15708.0,1429.0,097A,141010001097A,1,NaN,0,0,0
3,2.0,15708.0,1.0,1429.0,097A,141010001097A,1,NaN,0,0,0
4,3.0,2.0,3.0,2155.0,0090,141240090,1,7.0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
605321,511891.0,217423.0,217061.0,637.0,0062,140830062,1,9.0,0,0,0
605322,511892.0,217115.0,217424.0,636.0,0058,140830058,1,NaN,0,0,0
605323,511892.0,217424.0,217115.0,636.0,0058,140830058,1,8.0,0,0,0
605324,511893.0,217128.0,217425.0,639.0,0081,140830081,1,NaN,0,0,0


In [22]:
#links_from_visum = links_from_visum.drop(columns=['FROMNODENO', 'TONODENO'])
# llenar numéricas con 0
num_cols = ["main_road"]
links_from_visum[num_cols] = links_from_visum[num_cols].fillna(0)
links_from_visum

,No,from_node,to_node,ageb_df_idx,ageb_idx,ageb_code,topo_filt,main_road,connector_end_link,network_backbone,final_filter
0,1.0,1.0,103492.0,1436.0,1041,1410100011041,1,7.0,1,0,1
1,1.0,103492.0,1.0,1436.0,1041,1410100011041,1,0.0,1,0,1
2,2.0,1.0,15708.0,1429.0,097A,141010001097A,1,0.0,0,0,0
3,2.0,15708.0,1.0,1429.0,097A,141010001097A,1,0.0,0,0,0
4,3.0,2.0,3.0,2155.0,0090,141240090,1,7.0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
605321,511891.0,217423.0,217061.0,637.0,0062,140830062,1,9.0,0,0,0
605322,511892.0,217115.0,217424.0,636.0,0058,140830058,1,0.0,0,0,0
605323,511892.0,217424.0,217115.0,636.0,0058,140830058,1,8.0,0,0,0
605324,511893.0,217128.0,217425.0,639.0,0081,140830081,1,0.0,0,0,0


In [38]:
# Para todos los 605,326 de la red de Visum (.ver final) hubo coincidencia
new_cols = [
    "ageb_idx", "capacity", "num_lanes", "avg_vel", "limit_vel",
    "pop_dens", "job_dens", "dist_gdl", "boundary",
    "cluster", "subcluster", "topo_type", "topo_filt", "final_filter"
]
cols = ["RA_JERARQ", "GRUPO_CAP", "RA_CARGA", "NOMBRE_FIN"]

links_from_visum[cols].notna().sum()

RA_JERARQ     278121
GRUPO_CAP     277009
RA_CARGA      181197
NOMBRE_FIN    198850
dtype: int64

### 3) Load attribute/columns to visum

In [ ]:
visum_links = Visum.Net.Links

links =  links_from_visum.copy()
links = links.reset_index(drop=True)
links.index = links.index + 1

"""attributes = {
    "CAPACIDAD_FINAL": ("capacity", int),
    "CARRILES_FINAL": ("num_lanes", int),
    "VELPROM_FINAL": ("avg_vel", float),
    "LIMVEL_FINAL": ("limit_vel", float),
    "POP_DENSITY": ("pop_dens", float),
    "JOB_DENSITY": ("job_dens", float),
    "DIST_TO_GDL": ("dist_gdl", float),
    "IS_BOUNDARY": ("boundary", int),
    "CLUSTER": ("cluster", int),
    "SUBCLUSTER": ("subcluster", int),
    "TOPO_TYPE": ("topo_type", int),
    "TOPO_FILT": ("topo_filt", int),
    "AGEB_IDX": ("ageb_idx", int),
}"""

attributes = {
    "AGEB_DF_IDX": ("ageb_df_idx", str), 
    "AGEB_IDX": ("ageb_idx", str), 
    "AGEB_CODE": ("ageb_code", str), 
    "TOPO_FILT": ("topo_filt", int),
    "MAIN_ROAD": ("main_road", float), 
    "CONNECTOR_END_LINK": ("connector_end_link", int),
    "NETWORK_BACKBONE": ("network_backbone", int),
    "FINAL_FILTER": ("final_filter", int),
}


for visum_att, (df_col, dtype) in attributes.items():
    values = list(
        zip(
            links.index,
            links[df_col].astype(dtype)
        )
    )

    visum_links.SetMultiAttValues(visum_att, values)

In [7]:
links_from_visum

,No,from_node,to_node,final_filter
0,1.0,1.0,103492.0,1
1,1.0,103492.0,1.0,1
2,2.0,1.0,15708.0,1
3,2.0,15708.0,1.0,1
4,3.0,2.0,3.0,1
...,...,...,...,...
605321,511891.0,217423.0,217061.0,0
605322,511892.0,217115.0,217424.0,0
605323,511892.0,217424.0,217115.0,0
605324,511893.0,217128.0,217425.0,0


In [37]:
from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "from_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "to_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "CapPrT": [i[1] for i in Visum.Net.Links.GetMultiAttValues("CAPPRT")],
    "V0PrT": [i[1] for i in Visum.Net.Links.GetMultiAttValues("V0PRT")],
    "NumLanes": [i[1] for i in Visum.Net.Links.GetMultiAttValues("NUMLANES")],
    "highway": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")],
})

print(from_visum['CapPrT'].value_counts().to_string())

CapPrT
2000.0     321012
0.0        138143
1000.0      58371
4000.0      36253
3000.0      26573
1500.0       7697
6000.0       7090
4500.0       5002
2500.0       2212
5000.0        971
2.0           519
8000.0        443
3500.0        238
5.0           138
10000.0       135
3750.0        106
12000.0        94
2250.0         75
9000.0         74
1750.0         64
1250.0         56
800.0          25
7500.0         16
7.0            11
3250.0          8


In [56]:
pd.set_option('display.max_rows', None)

print(from_visum[from_visum['CapPrT']==0]['highway'].value_counts())

highway
                                                  96522
footway                                           21168
path                                               6160
cycleway                                           4195
track                                              3148
pedestrian                                         2756
steps                                               510
['footway', 'steps']                                493
['path', 'residential']                             428
['footway', 'residential']                          407
residential                                         305
['pedestrian', 'residential']                       207
['living_street', 'residential']                    196
['living_street', 'footway']                        176
corridor                                            150
['track', 'residential']                            137
service                                              93
unclassified                            